In [1]:
#Shishir Khanal
#Aug 24, 2026
#Lunar lander using REINFORCE algorithm
#Actions: Do nothing, Fire Left Engine, Fire main Engine, Fire Right Engine
#Positive Score - Soft landing

In [2]:
import gym
import pygame
import numpy as np
import torch as T
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
#Policy Network
# Input layer -> 128 neuron hidden layer -> Output layer
#Adam optimizer
#ReLu activation where appropriate, Output unactivated

class PolicyNetwork(nn.Module):
    def __init__(self, lr, input_dims, n_actions):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(*input_dims, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, n_actions)
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [4]:
#Agent: Constructor, action function and reward memory
#Agent needs gamma, policy Network, lists for memory of rewards, log probs
#PyTorch categorical distribution for action selection
class PolicyGradientAgent():
    def __init__(self, lr, input_dims, gamma=0.99, n_actions=4):
        self.gamma = gamma
        self.lr = lr
        self.reward_memory = []
        self.action_memory = []

        self.policy = PolicyNetwork(self.lr, input_dims, n_actions)

    def choose_action(self, observation):
        state = T.Tensor([observation]).to(self.policy.device)
        probabilities = F.softmax(self.policy.forward(state))
        action_probs = T.distributions.Categorical(probabilities)
        action = action_probs.sample()
        log_probs = action_probs.log_prob(action)
        self.action_memory.append(log_probs)

        return action.item()

    def store_rewards(self, reward):
        self.reward_memory.append(reward)

    def learn(self):
        self.policy.optimizer.zero_grad()
        #G_t = R_t+1 + gamma * R_t+2 + gamma**2 * R_t+3
        #G_t = sum from k=0 to k=T {gamma**k * R_t+k+1}
        G = np.zeros_like(self.reward_memory, dtype=np.float64)
        for t in range(len(self.reward_memory)):
            G_sum = 0
            discount = 1
            for k in range(t, len(self.reward_memory)):
                G_sum += self.reward_memory[k] * discount
                discount *= self.gamma
            G[t] = G_sum
        G = T.tensor(G, dtype=T.float).to(self.policy.device)

        loss = 0
        for g, logprob in zip(G, self.action_memory):
            loss += -g * logprob

        loss.backward()
        self.policy.optimizer.step()

        self.action_memory = []
        self.reward_memory = []

In [5]:
def plot_learning_curve(scores, x, figure_file):
    running_avg = np.zeros(len(scores))
    for i in range(len(running_avg)):
        running_avg[i] = np.mean(scores[max(0, i-100):(i+1)])
    plt.plot(x, running_avg)
    plt.title('Running average of previous 100 scores')
    plt.show()
    plt.savefig(figure_file)

In [ ]:
if __name__ == '__main__':
    render_every = 199  # set to 1 to watch every episode (slow)
    env = gym.make('LunarLander-v2', render_mode="rgb_array")
    n_games = 3000
    agent = PolicyGradientAgent(gamma=0.99, lr=0.0005, input_dims=[8], n_actions=4)
    fname = 'REINFORCE_' + 'lunar_lr' + str(agent.lr) + '_' + str(n_games) + 'games'
    figure_file = 'plots/' + fname + '.png'
    scores = []

    pygame.init()
    env.reset()  # must reset before the first render() call
    frame = env.render()  # grab one frame to get dimensions
    window = pygame.display.set_mode((frame.shape[1], frame.shape[0]))
    pygame.display.set_caption('LunarLander - REINFORCE')
    episode_font = pygame.font.SysFont('Arial', 24)
    clock = pygame.time.Clock()

    for i in range(n_games):
        done = False
        observation, info = env.reset()
        score = 0
        render_this_episode = (i % render_every == 0)

        while not done:
            action = agent.choose_action(observation)
            observation_, reward, truncated, terminated, info = env.step(action)
            done = truncated or terminated
            score += reward
            agent.store_rewards(reward)
            observation = observation_  # current state = new state

            if render_this_episode:
                pygame.event.pump()  # keep OS from thinking window is frozen
                frame = env.render()  # returns an (H, W, 3) rgb array
                surf = pygame.surfarray.make_surface(frame.swapaxes(0, 1))
                window.blit(surf, (0, 0))

                text_surface = episode_font.render(
                    f'REINFORCE Ep: {i + 1}/{n_games}', True, (255, 255, 255)
                )
                window.blit(text_surface, (10, 10))

                pygame.display.flip()
                clock.tick(60)

        agent.learn()
        scores.append(score)

        avg_score = np.mean(scores[-100:])
        print('episode', i, 'score %.2f' % score, 'average score %.2f' % avg_score)

    pygame.quit()
    x = [i + 1 for i in range(len(scores))]

/tmp/ipykernel_120801/4240493334.py:14: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  state = T.Tensor([observation]).to(self.policy.device)
/tmp/ipykernel_120801/4240493334.py:15: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probabilities = F.softmax(self.policy.forward(state))
/home/shishir/anaconda3/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


episode 0 score -115.19 average score -115.19
episode 1 score -144.75 average score -129.97
episode 2 score -173.76 average score -144.57
episode 3 score -376.22 average score -202.48
episode 4 score -316.14 average score -225.22
episode 5 score -166.61 average score -215.45
episode 6 score -73.49 average score -195.17
episode 7 score -349.01 average score -214.40
episode 8 score -113.06 average score -203.14
episode 9 score -105.85 average score -193.41
episode 10 score -174.07 average score -191.65
episode 11 score -159.41 average score -188.96
episode 12 score -156.94 average score -186.50
episode 13 score -160.19 average score -184.62
episode 14 score -285.30 average score -191.33
episode 15 score -97.72 average score -185.48
episode 16 score -252.82 average score -189.44
episode 17 score -438.93 average score -203.30
episode 18 score -376.13 average score -212.40
episode 19 score -119.38 average score -207.75
episode 20 score -291.32 average score -211.73
episode 21 score -303.45 

In [ ]:
# if __name__ == '__main__':
#     #env = gym.make('LunarLander-v2', render_mode="human")
#     env = gym.make('LunarLander-v2')
#     n_games = 3000
#     agent = PolicyGradientAgent(gamma=0.99, lr= 0.0005, input_dims=[8], n_actions=4)
#     fname = 'REINFORCE_' + 'lunar_lr' + str(agent.lr) + '_' + str(n_games) + 'games'
#     figure_file = 'plots/' + fname + '.png'
#     scores = []
#     for i in range(n_games):
#         done = False
#         observation, info = env.reset()
#         score = 0
#         while not done:
#             action = agent.choose_action(observation)
#             observation_, reward, truncated, terminated, info = env.step(action)
#             done = truncated or terminated
#             score += reward
#             agent.store_rewards(reward)
#             observation = observation_ #current state = new state
#         agent.learn()
#         scores.append(score)

#         avg_score = np.mean(scores[-100:])
#             #env.render()
#         print('episode', i, 'score %.2f' % score, 'average score %.2f' % avg_score)

#     x = [i+1 for i in range(len(scores))]


In [ ]:
    plot_learning_curve(scores, x, figure_file)